<a id="unit-u.c5d5db757ece41e485e9ae2930ee9466"></a>

<a id="ingest-observability-data-into-dog-with-opentelemetry"></a>

# Жүйені бақылау деректерін OpenTelemetry арқылы DOG-қа жүктеу

**2-модуль: Жүйені бақылау деректерін жүктеу** · [Сабақ](course.md) · [Тест](quiz02_ingesting_observability_data.ipynb)

1-модульде демо телеметриясы курс дайындаған конвейерлермен Doris-ке жетті. Бұл зертханалық жұмыста үш конвейерді өзіңіз құрасыз: әр сигналға бір OpenTelemetry Collector, әрқайсысы осы ноутбук жанындағы YAML файлымен бапталады. Бірі Doris FE журнал файлын оқиды, бірі Doris жариялайтын метрикаларды тартады, бірі Python бағдарламасы жіберген span-дарды қабылдайды. Әр сигналдың жеткенін Doris-тен тексересіз. Содан кейін 1-модульдегі бір сәтсіз тапсырысты Doris-ке дейін қадағалап, оның дәлелі қай кестелерде екенін және әр бөлікті қай конвейер жеткізгенін табасыз.

Шамамен 30 минут жоспарлаңыз. Төлем ақауы қосылған 1-модульдің DOG стегі мен демосы қажет (1-зертханалық жұмыстың 5-бөлімі).

<a id="unit-u.d29182be0cd3490587735f3067fc0c49"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.shell(...)` не `lab.sql(...)` қолданбас бұрын осы ұяшықты бір рет орындаңыз.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


<a id="unit-u.2ccfb921e56a4cd995f472c19c518b97"></a>

<a id="1-send-logs-to-dog"></a>

## 1. Журналдарды DOG-қа жіберу

Бірінші collector Doris FE-нің `fe.log` журналын оқып, әр жазбаны `otel_logs` кестесіне жазады. Конфигурациясы — [otel_for_logs.yaml](otel_for_logs.yaml):

- `filelog` қабылдағышы `/var/log/doris-fe/fe.log` файлын бірінші жолынан оқиды. Стек трассасы сияқты бір жазбаға тиесілі жолдарды біріктіріп, әр жазбаның уақыты мен маңыздылығын тұрақты өрнекпен талдайды.
- `resource` өңдегіші жазбалар дереккөзін атайды: `service.name` — `doris-fe`. Журнал файлы қай сервис жазғанын айтпайды.
- `batch` өңдегіші бірден 10,000 жазбаға дейін немесе 2 секундтан кейін жиналғанын жібереді.
- `doris` экспорттаушысы әр пакетті Stream Load арқылы `otel_logs` кестесіне жүктейді. Кесте құрмайды (`create_schema: false`): оларды 1-модульде DOG Collector-ы іске қосылғанда құрған.
- `file_storage` кеңейтімі қабылдағыштың оқу орнын, әр файлды қай жерге дейін оқығанын сақтайды.

Ұяшық Docker-ден FE журналдары хосттың қай каталогында екенін сұрап (`doris-obs` контейнері оларды `.runtime/doris-logs/fe/` ішіне жазады), каталогты collector-ға тек оқуға байланыстырады. Каталогты байланыстыру маңызды: FE журнал ротациясында `fe.log` атын өзгертіп, жаңа `fe.log` бастайды; тек ескі файл байланыстырылған collector жаңа жолдарды көрмес еді. Ұяшық оқу орны үшін `.runtime/collector-state/log2doris/` каталогын да байланыстырып, collector сол жерге жаза алуы үшін оны сіздің пайдаланушыңызбен орындайды. Қалған екі collector сияқты ол курстың `doris-obs` Docker желісіне қосылып, Doris-ке `doris-obs` контейнер атауы арқылы жетеді (YAML файлдарындағы `endpoint` қараңыз).

In [ ]:
lab.shell(r"""
set -euo pipefail

DORIS_CONTAINER="doris-obs"
CONFIG_DIR="level1/module02-data-ingest"
DORIS_FE_LOG_DIR="$(docker inspect --format '{{range .Mounts}}{{if eq .Destination "/opt/apache-doris/fe/log"}}{{.Source}}{{end}}{{end}}' "$DORIS_CONTAINER")"

if [ -z "$DORIS_FE_LOG_DIR" ]; then
  echo "Could not find the /opt/apache-doris/fe/log mount on the $DORIS_CONTAINER container."
  echo "Run module 1's prepare_environment() cell first, then rerun this cell."
  exit 1
fi

# Docker Desktop on macOS reports host mounts with a /host_mnt prefix.
DORIS_FE_LOG_DIR="${DORIS_FE_LOG_DIR#/host_mnt}"

if [ ! -f "$DORIS_FE_LOG_DIR/fe.log" ]; then
  echo "The Doris FE log was not found at $DORIS_FE_LOG_DIR/fe.log"
  exit 1
fi

# The collector saves how far it has read fe.log in this directory.
STATE_DIR="$PWD/.runtime/collector-state/log2doris"
mkdir -p "$STATE_DIR"

# Stop a running collector gracefully: it saves its read position and sends
# what it has read. Docker then removes it (--rm).
if docker container inspect doris-obs-log2doris >/dev/null 2>&1; then
  docker stop doris-obs-log2doris >/dev/null
  while docker container inspect doris-obs-log2doris >/dev/null 2>&1; do sleep 1; done
fi

# Run as your user, so that the collector can write its read position to STATE_DIR.
docker run --rm -d \
  --name doris-obs-log2doris \
  --network doris-obs \
  --user "$(id -u):$(id -g)" \
  -v "$PWD/$CONFIG_DIR/otel_for_logs.yaml:/etc/otel_for_logs.yaml:ro" \
  -v "$DORIS_FE_LOG_DIR:/var/log/doris-fe:ro" \
  -v "$STATE_DIR:/var/lib/otelcol/file_storage" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_logs.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-log2doris
""", title="Start the Doris log collector");

<a id="unit-u.46d645bea533451dbc316860c2a69dc4"></a>

**Күтілетін нәтиже**

Командалар журналы `collector=/doris-obs-log2doris status=running` жолымен аяқталады. Ұяшықты қайталағанда алдымен жұмыс істейтін collector тоқтатылады; ол оқу орнын сақтап, оқылған жазбаларды жібереді, ал жаңа collector сол орыннан жалғастырады. Демек ұяшықты қайталау FE журналын Doris-ке екінші рет жүктемейді.

<a id="unit-u.bf5f72c5bd82424ab2a4c20167a499c5"></a>

<a id="check-that-the-fe-logs-reached-doris"></a>

### FE журналдарының Doris-ке жеткенін тексеру

Жұмыс істейтін collector жолдардың Doris-ке жеткенін көрсетпейді. Келесі ұяшық оларды `otel_logs` кестесінен іздейді: filelog қабылдағышы файл атауын `log.file.name` атрибутына жазады, `otel_for_logs.yaml` ішіндегі `regex_parser` операторы әр жолдан `severity_text` толтырады, ал `resource` өңдегіші әр жолдың дереккөзін атайды: `service_name` — `doris-fe`. Пакет өңдегіші мен Doris-ке жүктеу бірнеше секунд алады, сондықтан ұяшық жолдар келгенше сұрауды қайталайды.

In [ ]:
fe_logs = lab.wait_for_rows(
    """
    SELECT service_name, severity_text, COUNT(*) AS log_rows, MAX(timestamp) AS latest_log
    FROM otel_logs
    WHERE log_attributes['log.file.name'] = 'fe.log'
    GROUP BY service_name, severity_text
    ORDER BY service_name, severity_text
    """,
    title="FE log rows in otel_logs",
    hint="Check that doris-obs-log2doris is running with docker ps, and read its log with docker logs doris-obs-log2doris.",
)

<a id="unit-u.4d2a1b87b8d34b5ea7133a3be870b5e7"></a>

**Күтілетін нәтиже**

FE журналындағы әр маңыздылық деңгейіне бір жол, барлығында `service_name = doris-fe`: әдетте `INFO` және `WARN`, ал FE қате жазса, `ERROR` да бар. Collector `fe.log` файлының ең ескі жолынан бастап, әр пакетте 10,000 жазбаға дейін экспорттайды (`otel_for_logs.yaml` ішіндегі `send_batch_size`), сондықтан іске қосылғаннан кейін бірден `latest_log` қазіргі уақыттан артта қалуы мүмкін. Біраздан соң ұяшықты қайталаңыз: `log_rows` өскен, ал `latest_log` қазіргі UTC уақытынан бірнеше секунд қана алшақ, өйткені FE `fe.log` файлына жаза береді, collector жаңа жолдарды жүктей береді.

<a id="unit-u.1c577a0e9c244faeb8ec748d161d22e9"></a>

<a id="where-the-table-settings-come-from"></a>

### Кесте баптаулары қайдан келеді

`otel_for_logs.yaml` бөлімдер не Doris жолдарды қанша уақыт сақтайтыны туралы ештеңе айтпайды. Экспорттаушы мұндай баптауларды кестені құрғанда ғана қолданады, ал `otel_logs` кестесін 1-модульдегі DOG Collector-ы өз конфигурациясымен құрған: `history_days: 7` және `create_history_days: 3`. Олар кестенің динамикалық бөлімдеуіне айналды: күніне бір бөлім алдын ала құрылып, 7 күннен асқанда жойылады. Телеметрия кестелерінің баптауларын тізіңіз:

In [ ]:
lab.sql(
    "SHOW DYNAMIC PARTITION TABLES FROM otel",
    title="Dynamic partitioning of the telemetry tables",
    columns=["TableName", "TimeUnit", "Start", "End", "Buckets", "ReplicationNum"],
);

<a id="unit-u.dbfc6f0442284c4984617c4533a50180"></a>

**Күтілетін нәтиже**

Сегіз жол, экспорттаушы құрған әр кестеге бірден (кейінгі модульдер құрған кестелер де қосылуы мүмкін); барлығында `TimeUnit` — `DAY`, `Start` — `-7`, `End` — `1`: Doris соңғы 7 күн бөлімдерін сақтап, келесі күн бөлімін алдын ала құрады. Осы зертханалық жұмыстың YAML файлындағы `history_days: 10` сияқты баптау ештеңе өзгертпейді, өйткені кестелер бұрыннан бар; экспорттаушының `create_schema` баптауы оларды мүлде құратынын анықтайды. 5-модуль бөлімдерді егжей-тегжейлі қарастырады.

<a id="unit-u.76ec94aa66c142c2a4ba58032545186c"></a>

<a id="2-send-metrics-to-dog"></a>

## 2. Метрикаларды DOG-қа жіберу

Екінші collector, [otel_for_metrics.yaml](otel_for_metrics.yaml), деректің келуін күтпейді: оның `prometheus` қабылдағышы деректі тартады. Әр 15 секунд сайын метрикаларын Prometheus мәтіні ретінде жариялайтын FE-ден (`doris-obs:8030`) және BE-ден (`doris-obs:8040`) `/metrics` сұрайды. Экспорттаушы әр метриканы түріне сай кестеге жазып, кесте атауларының префиксі ретінде `otel_metrics` қолданады.

In [ ]:
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-metric2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-metric2doris \
  --network doris-obs \
  -v "$PWD/$CONFIG_DIR/otel_for_metrics.yaml:/etc/otel_for_metrics.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_metrics.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-metric2doris
""", title="Start the Doris metric collector");

<a id="unit-u.02851803e9ea499b9c853d227701430a"></a>

**Күтілетін нәтиже**

Командалар журналы `collector=/doris-obs-metric2doris status=running` жолымен аяқталады.

<a id="unit-u.25038efe56834cce905040cc633fb0c9"></a>

<a id="check-that-the-metrics-reached-doris"></a>

### Метрикалардың Doris-ке жеткенін тексеру

Prometheus қабылдағышы `otel_for_metrics.yaml` ішіндегі екі нысанадан әр 15 секунд сайын метрика жинап, әр қатарды жинау тапсырмасымен (`service_name = 'doris_cluster'`) және нысанамен (`service_instance_id`) атайды. Doris экспорттаушысы әр метрика түріне бір кесте жазып, префикс ретінде бапталған `otel_metrics` атауын қолданады: gauge — `otel_metrics_gauge`, санауыш — `otel_metrics_sum` және әрі қарай. Келесі ұяшық екі нысананың да `otel_metrics_gauge` кестесінде жолдары болғанша күтеді.

In [ ]:
doris_metrics = lab.wait_for_rows(
    """
    SELECT service_instance_id AS scrape_target,
           COUNT(DISTINCT metric_name) AS metric_names,
           COUNT(*) AS data_points,
           MAX(timestamp) AS latest_point
    FROM otel_metrics_gauge
    WHERE service_name = 'doris_cluster'
    GROUP BY service_instance_id
    ORDER BY scrape_target
    """,
    title="Doris metrics in otel_metrics_gauge",
    hint="Check that doris-obs-metric2doris is running with docker ps, and read its log with docker logs doris-obs-metric2doris.",
    min_rows=2,
)

<a id="unit-u.a6e76a6ddba945d8962e98574171f342"></a>

**Күтілетін нәтиже**

Әдетте 30 секунд ішінде екі жол: `doris-obs:8030` (FE, шамамен 70 метрика атауы) және `doris-obs:8040` (BE, 100-ден астам). `latest_point` қазіргі UTC уақытына жақын, ал ұяшықты қайталағанда `data_points` әр жинаған сайын өседі.

Collector журналы (`docker logs doris-obs-metric2doris`) `doris_be_tablet_version_num_distribution` аталған бірнеше `warn` жолын да көрсетеді: BE бұл метриканы интервал шекаралары жоқ гистограмма ретінде жариялайды, сондықтан қабылдағыш оны тастайды. Қалған метрикалар қалыпты жүктеледі; журналдағы әр Stream Load жауабы `"NumberFilteredRows": 0` деп хабарлайды.

<a id="unit-u.dac1b28cd4114cc981952fcec5fe45b6"></a>

<a id="3-send-traces-to-dog"></a>

## 3. Трассаларды DOG-қа жіберу

Бұл collector span-дарды OTLP арқылы қабылдап, `deployment.environment.name` ресурс атрибутын қосады (`otel_for_traces.yaml` ішіндегі `resource` өңдегіші) және `otel_traces` кестесіне жазады. Қалған екі collector-дан айырмашылығы, келесі бөлімнің Python коды span жібере алуы үшін OTLP/HTTP қабылдағышын хостта `127.0.0.1:24328` адресінде (немесе `DORIS_OBS_TRACES_OTLP_HTTP_PORT` портында) ашады. DOG стегінің өз Collector-ы `127.0.0.1:24318` адресін тыңдайды; сонда жіберілген span-дар осы collector-дан және оның конфигурациясынан өтпейді.

Ұяшықтың бірінші жолы хост портын басқа бағдарлама қолданбайтынын тексереді.

In [ ]:
lab.check_trace_collector_port()
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-traces2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-traces2doris \
  --network doris-obs \
  -p "127.0.0.1:${DORIS_OBS_TRACES_OTLP_HTTP_PORT:-24328}:4318" \
  -v "$PWD/$CONFIG_DIR/otel_for_traces.yaml:/etc/otel_for_traces.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_traces.yaml

# Wait until the collector's log says that it is ready to receive data.
for attempt in $(seq 1 30); do
  docker logs doris-obs-traces2doris 2>&1 | grep "Everything is ready" >/dev/null && break
  sleep 1
done
docker logs doris-obs-traces2doris 2>&1 | grep -o "Everything is ready.*processing data."

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-traces2doris
docker port doris-obs-traces2doris
""", title="Start the trace collector");

<a id="unit-u.e47a7ab3a5e44590b53ac10aa498f15f"></a>

**Күтілетін нәтиже**

Порт тексеруі `24328` мәнін `available`, ал ұяшықты қайта орындағанда `used by this course` деп көрсетеді. Collector қабылдағышын бүкіл конвейері іске қосылған соң ашады, сондықтан ұяшық `Everything is ready. Begin running and processing data.` журнал жолын күтеді. Командалар журналы сол жолмен, `collector=/doris-obs-traces2doris status=running` және `4318/tcp -> 127.0.0.1:24328` мәндерімен аяқталады.

<a id="unit-u.b8c20e0b6d4c4b5e970bb676cf4d68a9"></a>

<a id="4-send-a-span-and-find-it-in-doris"></a>

## 4. Span жіберу және Doris-тен табу

Төмендегі код курстың `requirements.txt` файлы орнататын OpenTelemetry SDK көмегімен бір span құрып, оны OTLP/HTTP арқылы трасса collector-ына экспорттайды.

In [ ]:
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

resource = Resource.create({"service.name": "python-demo-service"})

provider = TracerProvider(resource=resource)
# The trace collector's OTLP/HTTP receiver: http://127.0.0.1:24328 unless DORIS_OBS_TRACES_OTLP_HTTP_PORT is set.
processor = BatchSpanProcessor(OTLPSpanExporter(endpoint=f"{lab.trace_collector_url}/v1/traces"))
provider.add_span_processor(processor)
# Use this provider directly. The global trace.set_tracer_provider() accepts only one
# provider per Python process, so a rerun of this cell would keep using the first one.
tracer = provider.get_tracer("demo-tracer")

with tracer.start_as_current_span("demo-span") as span:
    span.set_attribute("custom.attribute", "hello-otel")
    span.set_attribute("http.status_code", 200)

# BatchSpanProcessor exports in the background. Export the queued span now.
provider.force_flush()
trace_id = format(span.get_span_context().trace_id, "032x")
print(f"Flushed the span. Trace ID: {trace_id}")


<a id="unit-u.b5814d7f336c4b2e84e379f0f146061f"></a>

**Күтілетін нәтиже**

Ұяшық span-ның 32 таңбалы трасса ID-ін шығарады, мысалы `Flushed the span. Trace ID: 4bf92f3577b34da6a3ce929d0e0e4736`; сіздікі әр орындауда өзгереді. `force_flush()` экспорт әрекетін күтеді, бірақ нәтижесін хабарламайды: collector қолжетімсіз болса, экспорттаушы ұяшық астына тек `Failed to export spans batch` сияқты ескертулер жазады. Басып шығару да, flush та Doris span-ды сақтағанын көрсетпейді, сондықтан келесі ұяшық оны іздейді.

<a id="unit-u.87d6bcdc652e485fab6330410701bca5"></a>

<a id="find-the-span-in-doris"></a>

### Span-ды Doris-тен табу

Span-ды трасса ID-і бойынша іздеңіз. `environment` бағаны трасса collector-ы қосатын ресурс атрибутын оқиды. Python коды оны орнатпайды, сондықтан осы бағандағы `course-lab` мәні span-ның `doris-obs-traces2doris` арқылы өткенін көрсетеді.

In [ ]:
span_rows = lab.wait_for_rows(
    f"""
    SELECT trace_id, service_name, span_name, timestamp,
           resource_attributes['deployment.environment.name'] AS environment,
           span_attributes['custom.attribute'] AS custom_attribute
    FROM otel_traces
    WHERE trace_id = '{trace_id}'
    """,
    title="The span in otel_traces",
    hint="Check that doris-obs-traces2doris is running with docker ps, read its log with docker logs "
         "doris-obs-traces2doris, and look for export warnings under the previous cell.",
)

<a id="unit-u.418d29498dff4b6a85fae14176b13b5b"></a>

**Күтілетін нәтиже**

Әдетте 20 секунд ішінде бір жол: сіздің трасса ID-іңіз, `python-demo-service`, `demo-span`, UTC бойынша `timestamp`, `environment = course-lab` және `custom_attribute = hello-otel`. Кідіріс collector пакет өңдегішінен (`otel_for_traces.yaml` ішіндегі `timeout: 10s`) және Doris жүктеуді бекіткенше күтуден туады.

<a id="unit-u.2723bc5d13cb4e47a66e5563457249f1"></a>

<a id="try-it-change-the-collector-configuration"></a>

### Байқап көріңіз: collector конфигурациясын өзгерту

Collector `otel_for_traces.yaml` файлын тек іске қосылғанда оқиды.

1. `otel_for_traces.yaml` ішінде `value: course-lab` мәнін `value: my-laptop` сияқты басқа мәнге ауыстырып, файлды сақтаңыз.
2. Collector-ды жаңа файлмен қайта іске қосу үшін 3-бөлімдегі ұяшықты қайталаңыз. Ұяшық сәтсіз болса, collector өңделген файлмен іске қосыла алмады: өзгерісті тексеріңіз не кері қайтарып, ұяшықты қайталаңыз.
3. Осы бөлімдегі екі код ұяшығын қайта орындаңыз. Жаңа span `environment` ішінде сіздің мәніңізді көрсетеді.

Таңдаған мәніңіз төмендегі аяқтау тексеруіне де сәйкес келеді. Бастапқы мысалды қалпына келтіру үшін `otel_for_traces.yaml` ішіне `value: course-lab` мәнін қайтарып сақтаңыз, collector-ды қайта іске қосу үшін 3-бөлімді орындаңыз және 4-бөлімнің екі код ұяшығымен жаңа span жіберіңіз. Бұрынғы span-дар жіберілген мәндерін сақтайды.

<a id="unit-u.1a5918bbdcae4a6492e29f7b0eb9ee83"></a>

<a id="5-follow-one-failed-order-into-doris"></a>

## 5. Бір сәтсіз тапсырысты Doris-ке дейін қадағалау

1-модульде төлем ақауын қосқалы демо тапсырыстарының шамамен жартысы сәтсіз болады. Бір сәтсіз тапсырыстың дәлелі үш кестеге таралған. Ол жаңа құрған конвейерлеріңізге ұқсас жолмен келді: демо сервистері span, журнал және метрикаларын OTLP арқылы демоның өз Collector-ына жібереді, ол DOG Collector-ына бағыттайды, ал оның `doris` экспорттаушысы Doris-ке жүктейді.

Ең соңғы сәтсіз тапсырысты табыңыз. Checkout сервисі әр тапсырысты `oteldemo.CheckoutService/PlaceOrder` span-ы ретінде тіркейді; тапсырыс сәтсіз болса, span күйі `STATUS_CODE_ERROR` болады:

In [ ]:
failed_order = lab.wait_for_rows(
    """
    SELECT trace_id, timestamp
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 30 MINUTE
    ORDER BY timestamp DESC
    LIMIT 1
    """,
    title="The most recent failed order",
    hint="No order failed in the last 30 minutes. Turn on paymentFailure in section 5 of Lab 1, "
         "wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)
failed_trace = failed_order["trace_id"][0]

<a id="unit-u.52c17232eeff49f0a7e5755839451535"></a>

**Күтілетін нәтиже**

Бір жол: тапсырыс трассасының ID-і және оның checkout span-ының UTC уақыты. Келесі ұяшықтар осы трасса ID-ін әр кестеден іздейді.

<a id="unit-u.0d0ab4ff2fd0467ca06ae61e8c8096cf"></a>

<a id="the-spans-in-otel_traces"></a>

### `otel_traces` кестесіндегі span-дар

Тапсырыстың әр span-ында оның трасса ID-і бар. Span-дарды сервис бойынша және олардың қаншасында қате күйі барын санаңыз:

In [ ]:
lab.sql(f"""
SELECT service_name,
       COUNT(*) AS spans,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS error_spans
FROM otel_traces
WHERE trace_id = '{failed_trace}'
GROUP BY service_name
ORDER BY error_spans DESC, spans DESC
""", title="The spans of the failed order, by service");

<a id="unit-u.3ea1c9cc9769488fae4be551057f08a1"></a>

**Күтілетін нәтиже**

Шамамен он сервис және 25–40 span; саны тапсырыстағы тауар санына байланысты. Бес сервистің қате span-дары бар және олар алдымен тізіледі: `frontend` — үш, `checkout` пен `frontend-proxy` — екіден, `payment` пен `load-generator` — бірден. Бұлар тапсырыс жолындағы, жүктеме генераторынан дүкен проксиі мен frontend арқылы checkout пен payment-ке дейінгі сервистер. `currency`, `product-catalog`, `quote`, `cart`, `shipping` сияқты қалған сервистер жұмысын қатесіз орындады. Әр сервис OpenTelemetry SDK-сы өз span-дарын жіберді, ал ортақ трасса ID-і оларды бір сұрауға біріктіреді. 3-модуль трассаны Grafana-да ашып, қате басталған span-ды табады.

<a id="unit-u.263b24368e5d4c01ab88e7428223f9bc"></a>

<a id="the-logs-in-otel_logs"></a>

### `otel_logs` кестесіндегі журналдар

OpenTelemetry SDK әр журнал жазбасына сервис өңдеген сұраудың трасса ID-ін қояды, сондықтан сол ID тапсырыс журналдарын табады:

In [ ]:
lab.sql(f"""
SELECT timestamp, service_name, severity_number, severity_text, LEFT(body, 90) AS body_start
FROM otel_logs
WHERE trace_id = '{failed_trace}'
ORDER BY timestamp
""", title="The logs of the failed order");

<a id="unit-u.b615a114007148a4a524ac26866e730e"></a>

**Күтілетін нәтиже**

Бірнеше жол. Біріншісі — дүкен проксиінің қатынас журналы: `service_name` — `frontend-proxy`, ал мәтінінде `"POST /api/checkout HTTP/1.1" 500` бар. Қалғандары — өз жұмысын орындаған `cart`, `currency`, `quote` сияқты сервистердің `INFO` жазбалары. `payment` не `checkout` жолдары жоқ: бұл демода осы екеуі журналын тек контейнер шығысына жазады (`docker logs doris-obs-demo-payment` төлем қатесін көрсетеді), Collector-ға журнал жібермейді. Қатынас журналының маңыздылық деңгейі болмағандықтан, прокси жолының `severity_number` мәні 0, `severity_text` бос. Сондықтан қате журналдарын іздеу бұл ақауды өткізіп алады; 3-модуль оның орнына қатынас журналынан іздейді.

<a id="unit-u.cb484d7e363c4adeb2272b8f37986f3e"></a>

<a id="the-metrics-in-otel_metrics_sum"></a>

### `otel_metrics_sum` кестесіндегі метрикалар

Демо Collector-ы span-дарды метрикаға да айналдырады: оның `spanmetrics` коннекторы әр сервис, span атауы және күйі бойынша span-дарды санап, санақтарды `traces.span.metrics.calls` метрикасы ретінде жібереді. Санақ тек өседі, сондықтан ол sum түрінде `otel_metrics_sum` кестесіне түседі; әр нүкте коннектор санауды бастағаннан (`start_time`) бергі жиынтықты береді. Payment сервисінің төлем сұрауларының соңғы санақтарын оқыңыз:

In [ ]:
lab.sql("""
SELECT CAST(attributes['status.code'] AS STRING) AS status_code,
       MAX_BY(value, timestamp) AS charges_counted,
       MAX(start_time) AS counting_since,
       MAX(timestamp) AS latest_point
FROM otel_metrics_sum
WHERE metric_name = 'traces.span.metrics.calls'
  AND service_name = 'payment'
  AND attributes['span.name'] = 'grpc.oteldemo.PaymentService/Charge'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
GROUP BY status_code
ORDER BY status_code
""", title="Charge requests counted by the demo Collector, by status");

<a id="unit-u.e111ef4d156a4ff7b967d7861a687f67"></a>

**Күтілетін нәтиже**

Екі жол: сәтсіз төлем сұраулары — `STATUS_CODE_ERROR`, сәттілері — `STATUS_CODE_UNSET`; екеуі де демо басталған `counting_since` уақытынан саналады. Ақаулар жалаушаны қосқанда басталды. Санауыш кез келген уақыт масштабында «қанша» және «қаншалықты жиі» сұрақтарына арзан жауап береді, бірақ ешбір нүктесі жеке тапсырысқа апармайды: тапсырыс span-дары тек трассада бар.

| Кесте | Сәтсіз тапсырыстың дәлелі | Қалай жетті |
| --- | --- | --- |
| `otel_traces` | `payment` пен оның үстіндегі сервистерде қате күйі бар тапсырыс span-дары | Әр сервис SDK-сы, кейін демо Collector-ы мен DOG Collector-ы |
| `otel_logs` | Проксидің HTTP 500 қатынас журналы және басқа сервистердің `INFO` жазбалары; `payment` не `checkout` дерегі жоқ | Сол жолмен |
| `otel_metrics_sum` | Тапсырыстың өзі емес, сәтсіз және сәтті төлем сұрауларының саны | Демо Collector-ының `spanmetrics` коннекторы span-дардан жасайды |

<a id="unit-u.592a648e932e40069b73971060c6bb9c"></a>

<a id="6-exercise-count-the-failed-checkouts-in-the-access-log"></a>

## 6. Жаттығу: қатынас журналынан сәтсіз checkout санын есептеу

Дүкен проксиі арқылы өткен әр сұрау `otel_logs` ішінде сұраудан кейінгі HTTP күйі бар бір қатынас журналы жолын қалдырады, мысалы `"POST /api/checkout HTTP/1.1" 500`. Дүкен бетінде берілген тапсырыста жолдан кейін сұрау параметрлері болады, мысалы `"POST /api/checkout?currencyCode=USD HTTP/1.1" 500`. Соңғы 15 минуттағы checkout сұрауларын HTTP күйі бойынша санайтын сұрау жазыңыз. Кеңес: `REGEXP_EXTRACT(body, '<pattern>', 1)` `body` ішінен үлгідегі бірінші топқа сәйкес бөлікті қайтарады.

In [ ]:
# Your code here

<a id="unit-u.585f1555bd064cfb8f059458bf6a9c52"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
lab.sql("""
SELECT REGEXP_EXTRACT(body, '"POST /api/checkout[^"]*" ([0-9]+)', 1) AS http_status,
       COUNT(*) AS requests
FROM otel_logs
WHERE service_name = 'frontend-proxy'
  AND body LIKE '%"POST /api/checkout%'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
GROUP BY http_status
ORDER BY http_status
""", title="Checkout requests of the last 15 minutes, by HTTP status");

<a id="unit-u.85b4f90006894cc5ac709b6eb827cc81"></a>

**Күтілетін нәтиже**

Екі жол: өткен тапсырыстар үшін `200`, сәтсіздері үшін `500`. Жалаушаны қосқаннан кейін сұраулардың шамамен жартысы `500`; тапсырыс аз болғандықтан, үлес жартыдан едәуір жоғары не төмен болуы мүмкін. Жалаушаға дейінгі сұраулардың бәрі `200`. Әр `500` қате күйі бар checkout span-ына сәйкес келеді, сондықтан қатынас журналы мен трассалар бір ақауларды санайды.

<a id="unit-u.70a5c4feb1ea4e918a67669e3981c345"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- `otel_logs` кестесінде `service_name = doris-fe` мәнді FE журналы бар, ал 1-бөлімді қайталау оны қайта жүктемейді;
- `otel_metrics_gauge` кестесінде Doris-тің екі жинау нысанасының да дерек нүктелері бар;
- span-ыңыз `otel_traces` ішінде өзіңіз таңдаған `environment` мәнімен бар (тәжірибеге дейін `course-lab`);
- бір сәтсіз тапсырыстың span-дарын, журналдарын және метрика санақтарын таптыңыз.

Үш collector жұмысын жалғастырады және жадыны аз қолданады; кейінгі модульдерге олар қажет емес, `lab.teardown()` оларды курстың қалған ортамен бірге жояды. Ертерек тоқтату үшін терминалда `docker stop doris-obs-log2doris doris-obs-metric2doris doris-obs-traces2doris` орындаңыз.

<a id="unit-u.0077be27dfd04a04bfbdc475c48079fb"></a>

<a id="next"></a>

## Әрі қарай

[2-тест: Жүйені бақылау деректерін жүктеу](quiz02_ingesting_observability_data.ipynb) файлын ашып, [3-модуль: Жүйені бақылау деректерін визуализациялау](../module03-data-visualization/course.md) бөліміне өтіңіз.